In [66]:
# data analysis
import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
from pyampute.exploration.md_patterns import mdPatterns
from pyampute.exploration.mcar_statistical_tests import MCARTest
import missingno as msno


# preprocessing
import sklearn.utils.validation
import sys
from scipy import stats
from scipy.stats import shapiro, distributions, loguniform
from scipy.stats.mstats import winsorize
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, GridSearchCV, HalvingRandomSearchCV, HalvingGridSearchCV, TunedThresholdClassifierCV, FixedThresholdClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, QuantileTransformer, MinMaxScaler, KBinsDiscretizer, Binarizer, PolynomialFeatures, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from feature_engine.outliers import Winsorizer
# from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.pipeline import Pipeline
from sklearn import set_config

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# Modeling
from sklearn.linear_model import RidgeClassifier, LogisticRegression, RidgeClassifierCV, LogisticRegressionCV, SGDClassifier, Perceptron, PassiveAggressiveClassifier
from sklearn.svm import LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
import joblib
from sklearn import tree 
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier, VotingClassifier, StackingClassifier
from lightgbm import LGBMClassifier 

# Metrics
from sklearn.metrics import confusion_matrix, recall_score, precision_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report, precision_recall_curve, PrecisionRecallDisplay, log_loss, brier_score_loss, roc_curve, roc_auc_score, RocCurveDisplay, det_curve, DetCurveDisplay, fbeta_score, average_precision_score, matthews_corrcoef

# Calibration
from sklearn.calibration import calibration_curve, CalibrationDisplay, CalibratedClassifierCV

# Inspection
from sklearn.inspection import PartialDependenceDisplay

# Custom Functions
from credit_risk_modeling import model_eval
import importlib
importlib.reload(model_eval)

<module 'credit_risk_modeling.model_eval' from 'C:\\Users\\billy\\OneDrive\\Documents\\Finance_Projects\\credit_risk_modeling\\credit_risk_modeling\\model_eval.py'>

## Imports

In [67]:
X_train = pd.read_csv(
    filepath_or_buffer = "../data/processed/X_train_tree.csv"
)
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22686 entries, 0 to 22685
Data columns (total 8 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   numeric__person_age                         22686 non-null  float64
 1   numeric__person_income                      22686 non-null  float64
 2   numeric__person_emp_length                  22686 non-null  float64
 3   numeric__loan_amnt                          22686 non-null  float64
 4   numeric__loan_int_rate                      22686 non-null  float64
 5   numeric__loan_percent_income                22686 non-null  float64
 6   numeric__cb_person_cred_hist_length         22686 non-null  float64
 7   categorical__cb_person_default_on_file_1.0  22686 non-null  float64
dtypes: float64(8)
memory usage: 1.4 MB


In [68]:
X_test = pd.read_csv(
    filepath_or_buffer = "../data/processed/X_test_tree.csv"
)
X_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9723 entries, 0 to 9722
Data columns (total 8 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   numeric__person_age                         9723 non-null   float64
 1   numeric__person_income                      9723 non-null   float64
 2   numeric__person_emp_length                  9723 non-null   float64
 3   numeric__loan_amnt                          9723 non-null   float64
 4   numeric__loan_int_rate                      9723 non-null   float64
 5   numeric__loan_percent_income                9723 non-null   float64
 6   numeric__cb_person_cred_hist_length         9723 non-null   float64
 7   categorical__cb_person_default_on_file_1.0  9723 non-null   float64
dtypes: float64(8)
memory usage: 607.8 KB


In [69]:
y_train = pd.read_csv(
    filepath_or_buffer = "../data/interim/y_train.csv"
)
y_train = y_train.values.ravel()

In [70]:
y_test = pd.read_csv(
    filepath_or_buffer = "../data/interim/y_test.csv"
)
y_test.head(1)
y_test = y_test.values.ravel()

## Compare Models

Not accouting for feature selection (unless), class imbalance, or hyperparameter tuning

In [71]:
untuned_models = [
    tree.DecisionTreeClassifier(
        random_state = 42,
    ), # no sampling 
    AdaBoostClassifier(
        random_state=42
    ), # default estimator is DecisionTreeClassifier, default no sampling
    XGBClassifier(
        reg_lambda=1,
        reg_alpha=0,
        random_state=42
    ), # default embedded l2 regularization, default includes no sampling (but sampling is possible for both samples and features - would be pasting + subspacing so replacement=None)
    LGBMClassifier(
    objective='binary',
    reg_alpha = 0,
    reg_lambda = 1,
    random_state=42
    ), # default embedded l2 regularization, default includes no sampling (but sampling is possible for both samples and features - would be pasting + subspacing so replacement=None)
    HistGradientBoostingClassifier(
        l2_regularization= 0,
        random_state=42
    ), # no regularization, default no subsampling but can be turned on
    GradientBoostingClassifier(
        subsample = 1.0,
        random_state=42
    ), # default no subsampling regularization but Pasting can be turned on
    RandomForestClassifier(
        n_jobs=-1,
        random_state=42
    ), # default Bagging, Subspacing
    ExtraTreesClassifier(
        n_jobs=-1,
        random_state=42
    ), # default all samples but Bagging can be turned on, Subspacing for features
    BaggingClassifier(estimator=tree.DecisionTreeClassifier(
        random_state=42), 
        n_estimators=50, 
        bootstrap=True, 
        max_samples=1.0,
        max_features=1.0, 
        random_state=42),
    VotingClassifier(
        estimators= [('gb', GradientBoostingClassifier(random_state=42)), ('rf', RandomForestClassifier(random_state=42)), ('xgb', XGBClassifier(random_state=42))],
        n_jobs=-1
    ),
    StackingClassifier(
        estimators= [('gb', GradientBoostingClassifier(random_state=42)), ('rf', RandomForestClassifier(random_state=42))],
        final_estimator= ('xgb', XGBClassifier(random_state=42)),
        n_jobs=-1
    ),
    StackingClassifier(
        estimators= [('gb', GradientBoostingClassifier(random_state=42)), ('rf', RandomForestClassifier(random_state=42)), 'xgb', XGBClassifier(random_state=42)],
        final_estimator= StackingClassifier(
            estimators= [('gb', HistGradientBoostingClassifier(random_state=42)), ('rf', ExtraTreesClassifier(random_state=42))],
            final_estimator= ('xgb', LGBMClassifier(random_state=42)),
            n_jobs=-1
        ),
        n_jobs=-1
    ) # multiple layer
]

In [72]:
model_eval.__file__

'C:\\Users\\billy\\OneDrive\\Documents\\Finance_Projects\\credit_risk_modeling\\credit_risk_modeling\\model_eval.py'

In [ ]:

untuned_model_performance, fitted_models_internal = model_eval.comparing_models(untuned_models, X_train, y_train, X_test, y_test)

[LightGBM] [Info] Number of positive: 4962, number of negative: 17724
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000398 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1061
[LightGBM] [Info] Number of data points in the train set: 22686, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.218725 -> initscore=-1.273111
[LightGBM] [Info] Start training from score -1.273111
